In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import math

In [ ]:
TRAIN_DATA_PATH = r"/kaggle/input/house-prices-advanced-regression-techniques/train.csv"
TEST_DATA_PATH = r"/kaggle/input/house-prices-advanced-regression-techniques/test.csv"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
def load_data(path: str) -> pd.DataFrame:
    if not os.path.exists(path):
        raise FileNotFoundError(f"The file {path} does not exist.")

    df = pd.read_csv(path)
    print(f"Data loaded successfully from {path}.")
    print(f"Data shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")

    return df

In [ ]:
train_df = load_data(TRAIN_DATA_PATH)
print('-'*100)
test_df = load_data(TEST_DATA_PATH)

In [ ]:
train_df = train_df.drop(columns=['Id'])
x_test_id = test_df["Id"].copy()  
X_test = test_df.drop(columns=["Id"]) 
print('-'*100)
print('shape of train data:', train_df.shape)

In [ ]:
def missing_columns_control(df: pd.DataFrame) -> pd.DataFrame:
    missing_data = df.isnull().sum()
    missing_data = missing_data[missing_data > 0].sort_values(ascending=False)
    missing_percentage = (missing_data / len(df)) * 100
    missing_df = pd.DataFrame({'Missing Values': missing_data, 'Percentage': missing_percentage})
    
    return missing_df

def missing_values_handling(df: pd.DataFrame) -> pd.DataFrame:
    for column in df.columns:
        if df[column].dtype == 'object':
            df[column] = df[column].fillna(df[column].mode()[0])
            print(f"Filled missing values in categorical column '{column}' with mode: {df[column].mode()[0]}")
        else:
            df[column] = df[column].fillna(df[column].mean())  
            print(f"Filled missing values in numerical column '{column}' with mean: {df[column].mean()}")
    return df

def remove_highly_missing_columns(df: pd.DataFrame, threshold: float = 50.0) -> pd.DataFrame:
    missing_df = missing_columns_control(df)
    high_missing_cols = missing_df[missing_df['Percentage'] > threshold].index
    df = df.drop(columns=high_missing_cols)
    print(f"Removed columns with more than {threshold}% missing values: {list(high_missing_cols)}")
    return df

def plot_missing_value_per_column(missing_df):
    plt.figure(figsize=(10, 6))
    sns.barplot(x=missing_df.index, y='Percentage', data=missing_df)
    plt.xticks(rotation=90)
    plt.title('Missing Value Percentage per Column')
    plt.ylabel('Percentage (%)')
    plt.xlabel('Columns')
    plt.show()

def remove_columns_for_test(train_df: pd.DataFrame, test_df: pd.DataFrame) -> pd.DataFrame:
    train_cols = set(train_df.columns)
    test_cols = set(test_df.columns)
    
    cols_to_remove_from_test = test_cols - train_cols
    
    if cols_to_remove_from_test:
        test_df = test_df.drop(columns=cols_to_remove_from_test)
        print(f"Removed columns from test data not in train data: {list(cols_to_remove_from_test)}")
    
    return test_df

In [ ]:
train_missing_df = missing_columns_control(train_df)
print(train_missing_df)

In [ ]:
plot_missing_value_per_column(train_missing_df)

In [ ]:
removed_high_missing_train_df = remove_highly_missing_columns(train_df, threshold=60.0)

In [ ]:
test_df = remove_columns_for_test(removed_high_missing_train_df, test_df)

In [ ]:
print('-'*100)
def check_columns(df1: pd.DataFrame, df2: pd.DataFrame):
    cols1 = set(df1.columns)
    cols2 = set(df2.columns)
    
    only_in_df1 = cols1 - cols2
    only_in_df2 = cols2 - cols1
    
    if only_in_df1:
        print(f"Columns only in first DataFrame: {list(only_in_df1)}")
    else:
        print("No columns are unique to the first DataFrame.")
    
    if only_in_df2:
        print(f"Columns only in second DataFrame: {list(only_in_df2)}")
    else:
        print("No columns are unique to the second DataFrame.")

In [ ]:
print('-'*100)
check_columns(removed_high_missing_train_df, test_df)

In [ ]:
cleaned_train_df = missing_values_handling(removed_high_missing_train_df)
print('-'*100)
print('shape of cleaned train data:', cleaned_train_df.shape)

In [ ]:
def outlier_detection_iqr(df: pd.DataFrame):
    numeric_columns = df.select_dtypes(include=[np.number]).columns
    outlier_flags = pd.DataFrame(0, index=df.index, columns=numeric_columns)

    for column in numeric_columns:
        
        Q1 = df[column].quantile(0.25)
        Q3 = df[column].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR

        outlier_flags[column] = ((df[column] < lower_bound) | (df[column] > upper_bound)).astype(int)

    col_outlier_counts = outlier_flags.sum(axis=0)

    row_outlier_counts = outlier_flags.sum(axis=1)

    row_outliers = row_outlier_counts[row_outlier_counts > 0]

    return outlier_flags, col_outlier_counts, row_outliers


def plot_outlier_for_columns_grid(df: pd.DataFrame, kind="box", cols=4):
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    n = len(numeric_columns)
    rows = math.ceil(n / cols)

    fig, axes = plt.subplots(rows, cols, figsize=(cols*4, rows*3))
    axes = axes.flatten()

    for i, column in enumerate(numeric_columns):
        if kind == "box":
            sns.boxplot(y=df[column].dropna(), color="skyblue", ax=axes[i])
            axes[i].set_title(f"{column}")
        elif kind == "hist":
            sns.histplot(df[column].dropna(), kde=True, color="salmon", ax=axes[i])
            axes[i].set_title(f"{column}")
    
    for j in range(i+1, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()

    
    
def drop_rows_above_iqr(df: pd.DataFrame, row_counts: pd.Series, k: float = 1.5) -> pd.DataFrame:
    Q1 = row_counts.quantile(0.25)
    Q3 = row_counts.quantile(0.75)
    IQR = Q3 - Q1
    threshold = Q3 + k * IQR
    rows_to_drop = row_counts[row_counts > threshold].index
    print(f"Eşik: {threshold:.2f} (Q3 + {k}*IQR), silinecek satır: {len(rows_to_drop)}")
    return df.drop(index=rows_to_drop)


In [ ]:
flags, col_counts, row_counts = outlier_detection_iqr(cleaned_train_df)

print("---- Column bazlı outlier sayısı ----")
print(col_counts)

print("\n---- Row bazlı outlier sayısı ----")
print(row_counts.head(20))

In [ ]:
droped_outlier_train_df = drop_rows_above_iqr(cleaned_train_df, row_counts)
print('-'*100)
print('shape of train data after dropping outlier rows:', droped_outlier_train_df.shape)

In [ ]:
plot_outlier_for_columns_grid(cleaned_train_df, 'box')

In [ ]:
plot_outlier_for_columns_grid(droped_outlier_train_df, 'box')

In [ ]:
from sklearn.preprocessing import OneHotEncoder


def fit_onehot_encoder(df: pd.DataFrame):
    categorical_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
    df_final = df.copy()
    encoder = None
    
    if categorical_cols:
        print(f"Encoding {len(categorical_cols)} categorical columns: {categorical_cols}")
        encoder = OneHotEncoder(sparse_output=False, drop=None, handle_unknown="ignore")
        encoded = encoder.fit_transform(df[categorical_cols])
        encoded_cols = encoder.get_feature_names_out(categorical_cols)
        df_encoded = pd.DataFrame(encoded, columns=encoded_cols, index=df.index)
        
        df_final = pd.concat([df.drop(columns=categorical_cols), df_encoded], axis=1)
    else:
        print("No categorical columns found.")
    
    return df_final, encoder, categorical_cols

In [ ]:
df_train, encoder, categorical_cols = fit_onehot_encoder(droped_outlier_train_df) 
print('-'*100)
print('shape of train data after one-hot encoding:', df_train.shape)

In [ ]:
def transform_with_encoder(df: pd.DataFrame, encoder: OneHotEncoder, categorical_cols: list) -> pd.DataFrame:
    if encoder is None or not categorical_cols:
        print("No categorical columns to encode or encoder is None. Returning original DataFrame.")
        return df.copy()
    
    existing_cats = [col for col in categorical_cols if col in df.columns]
    if not existing_cats:
        print("No matching categorical columns found in the DataFrame. Returning original DataFrame.")
        return df.copy()
    
    # transform et
    encoded = encoder.transform(df[existing_cats])
    encoded_cols = encoder.get_feature_names_out(existing_cats)
    df_encoded = pd.DataFrame(encoded, columns=encoded_cols, index=df.index)
    
    # encode edilen kolonları drop et ve yeni encoded kolonları ekle
    df_final = pd.concat([df.drop(columns=existing_cats), df_encoded], axis=1)
    
    return df_final


In [ ]:
x_test = transform_with_encoder(test_df, encoder, categorical_cols)

In [ ]:
check_columns(df_train, x_test)

In [ ]:
y_train = df_train['SalePrice']
df_train = df_train.drop(columns=['SalePrice'])
print(df_train.columns)

In [ ]:
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.preprocessing import StandardScaler, MinMaxScaler, MaxAbsScaler, PowerTransformer, QuantileTransformer, RobustScaler
from sklearn.metrics import mean_squared_error
import numpy as np

def objective(trial):
    scalers = {
        "StandardScaler": StandardScaler(),
        "MinMaxScaler": MinMaxScaler(),
        "RobustScaler": RobustScaler(),
        "MaxAbsScaler": MaxAbsScaler()
    }

    scaler_choice = trial.suggest_categorical("scaler", list(scalers.keys()))
    scaler = scalers[scaler_choice]
    X_train_scaled = scaler.fit_transform(df_train).astype(np.float32)  
    y_train_np = y_train.values.astype(np.float32)

    params = {
        "objective": "reg:squarederror",
        "eval_metric": "rmse",
        "tree_method": "hist",   # GPU uyumlu
        "device": "cuda",        # GPU kullan
        "max_depth": trial.suggest_int("xgb_max_depth", 3, 15),
        "learning_rate": trial.suggest_float("xgb_learning_rate", 1e-3, 0.3, log=True),
        "subsample": trial.suggest_float("xgb_subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("xgb_colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("xgb_gamma", 0, 10.0),
        "reg_alpha": trial.suggest_float("xgb_reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("xgb_reg_lambda", 1e-8, 10.0, log=True),
        "min_child_weight": trial.suggest_int("xgb_min_child_weight", 1, 20),
        "seed": RANDOM_STATE,
    }
    n_estimators = trial.suggest_int("xgb_n_estimators", 200, 2000)

    kf = KFold(n_splits= 10, shuffle=True, random_state=RANDOM_STATE)
    rmse_score = []

    for train_idx, valid_idx in kf.split(X_train_scaled):
        X_tr, X_val = X_train_scaled[train_idx], X_train_scaled[valid_idx]
        y_tr, y_val = y_train_np[train_idx], y_train_np[valid_idx]

        dtrain = xgb.DMatrix(X_tr, label=y_tr)
        dvalid = xgb.DMatrix(X_val, label=y_val)

        booster = xgb.train(
            params,
            dtrain,
            num_boost_round=n_estimators,
            evals=[(dvalid, "valid")],
            early_stopping_rounds=50,
            verbose_eval=False
        )

        preds = booster.predict(dvalid)
        rmse = mean_squared_error(y_val, preds, squared=False)
        rmse_score.append(rmse)

    return np.mean(rmse_score)

In [ ]:
import optuna

study = optuna.create_study(
    direction="minimize",
    study_name="XGB_Regressor_Optimization"
)

study.optimize(objective, n_trials=100, show_progress_bar=True)

print('Training Complete!')
print(f'Total Number of Trials : {len(study.trials)}\n')

print("Best Trial:", study.best_trial.number)
print("Best RMSE:", study.best_value)
print("Best Params:", study.best_trial.params)

In [ ]:
from xgboost import XGBRegressor

scalers = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler": MinMaxScaler(),
    "RobustScaler": RobustScaler(),
    "MaxAbsScaler": MaxAbsScaler(),
    "QuantileTransformer": QuantileTransformer(output_distribution='normal'),
    "PowerTransformer": PowerTransformer()
}

scaler_key = study.best_trial.params['scaler']
del study.best_trial.params['scaler']

scaler = scalers[scaler_key]

best_params = study.best_trial.params

final_model = XGBRegressor(
    objective="reg:squarederror",
    eval_metric="rmse",
    tree_method="hist",
    device="cuda",
    max_depth=best_params["xgb_max_depth"],
    learning_rate=best_params["xgb_learning_rate"],
    subsample=best_params["xgb_subsample"],
    colsample_bytree=best_params["xgb_colsample_bytree"],
    gamma=best_params["xgb_gamma"],
    reg_alpha=best_params["xgb_reg_alpha"],
    reg_lambda=best_params["xgb_reg_lambda"],
    min_child_weight=best_params["xgb_min_child_weight"],
    n_estimators=best_params["xgb_n_estimators"],
    seed=RANDOM_STATE
)

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error
 
def eval_plot(model, X_train, y_train, scaler):
    scaled_X_train = scaler.fit_transform(X_train).astype(np.float32)
    y_train_np = y_train.values.astype(np.float32)

    model = model.fit(scaled_X_train, y_train_np)
    
    preds = model.predict(scaled_X_train)
    
    plt.figure(figsize=(8, 6))
    plt.scatter(y_train_np, preds, alpha=0.5, label="Samples")
    plt.plot([y_train_np.min(), y_train_np.max()],
             [y_train_np.min(), y_train_np.max()],
             'r--', label="Perfect Prediction")
    plt.xlabel('Actual Tm')
    plt.ylabel('Predicted Tm')
    plt.title('Actual vs Predicted Tm')
    plt.legend()
    plt.show()

    print(f"Train RMSE: {mean_squared_error(y_train_np, preds, squared=False):.4f}")
    print(f'Train MSE Score: {mean_squared_error(y_train_np, preds):.4f}')
    print(f'Train MAE Score: {mean_absolute_error(y_train_np, preds):.4f}')
    print(f'Train R2 Score: {r2_score(y_train_np, preds):.4f}')
    print(f'n samples: {len(y_train_np)}')

In [ ]:
def test_and_save(model, X_train, X_test, y_train, scaler, x_test_ids, to_save_path="submission.csv"):    
    X_train_scaled = scaler.fit_transform(X_train).astype(np.float32)
    y_train_np = y_train.reset_index(drop=True).values.astype(np.float32)

    model = model.fit(X_train_scaled, y_train_np)
    
    X_test_scaled = scaler.transform(X_test).astype(np.float32)
    
    y_preds = model.predict(X_test_scaled)
    
    to_save = pd.DataFrame({
        "Id": x_test_ids,
        "SalePrice": y_preds
    })
    
    to_save.to_csv(to_save_path, index=False)
    print(f"✅ Predictions saved to {to_save_path}")


In [ ]:
eval_plot(final_model, df_train, y_train, scaler)

In [ ]:
test_and_save(final_model, df_train, x_test, y_train, scaler, x_test_id, to_save_path="/kaggle/working/submission.csv")

In [ ]:
optuna.visualization.plot_optimization_history(study)

In [ ]:
optuna.visualization.plot_parallel_coordinate(study)

In [ ]:
optuna.visualization.plot_slice(study)

In [ ]:
fig = optuna.visualization.plot_contour(study)
fig.update_layout(width = 2000, height = 900)

In [ ]:
optuna.visualization.plot_edf(study)

In [ ]:
optuna.visualization.plot_param_importances(study)